# All-weather risk parity (inverse-vol)

Bridgewater-style all-weather, retail flavor. Five sleeves intended to span the four economic regimes (growth ↑/↓, inflation ↑/↓):

- **SPY** — US equities (growth ↑, inflation ↓)
- **TLT** — long-duration USTs (growth ↓, inflation ↓)
- **IEF** — intermediate USTs (growth ↓, inflation ↓ — softer version)
- **GLD** — gold (growth ↓, inflation ↑)
- **DBC** — broad commodities (growth ↑, inflation ↑)

Each month, set weights inversely proportional to trailing 63-day realized vol so each sleeve contributes a similar amount of risk. Sleeves sum to 1 (long-only, no leverage). Reuses `alpha_lab.portfolio.long_only.rolling_inverse_volatility_weights`.

In [ ]:
import pandas as pd

from alpha_lab.backtest.metrics import summary
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.data.loaders.yfinance import load_prices
from alpha_lab.portfolio.long_only import fixed_weight_returns, rolling_inverse_volatility_weights
from alpha_lab.reporting.charts import drawdown_chart, equity_curve, heatmap_monthly

UNIVERSE = ["SPY", "TLT", "IEF", "GLD", "DBC"]
START = "2007-01-01"           # DBC inception Feb 2006
END = None
LOOKBACK_DAYS = 63              # ~3 months of trading days
REBAL = "ME"
COSTS_BPS = 1.0
SLIPPAGE_BPS = 2.0

In [ ]:
prices = load_prices(UNIVERSE, start=START, end=END).dropna(how="any")
prices.tail()

In [ ]:
w_monthly = rolling_inverse_volatility_weights(prices, lookback_days=LOOKBACK_DAYS, rebalance=REBAL)
weights = w_monthly.reindex(prices.index).ffill().fillna(0.0)
w_monthly.tail()

In [ ]:
# Average weights over the sample -- shows where risk-parity actually lands vs naive 1/N.
w_monthly.mean().rename("avg_weight").to_frame().assign(naive=1.0 / len(UNIVERSE))

In [ ]:
res = run_backtest(weights, prices, rebalance=REBAL, costs_bps=COSTS_BPS, slippage_bps=SLIPPAGE_BPS)

rets = prices.pct_change().fillna(0.0)
bench = pd.DataFrame({
    "risk parity": res.returns,
    "equal-weight (1/5)": fixed_weight_returns(rets, {t: 1.0 / len(UNIVERSE) for t in UNIVERSE}),
    "60/40 (SPY/IEF)": fixed_weight_returns(rets, {"SPY": 0.6, "IEF": 0.4}),
    "BH SPY": rets["SPY"],
}).dropna()
pd.DataFrame({name: pd.Series(summary(s)) for name, s in bench.items()})

In [ ]:
annual_turnover = res.turnover.resample("YE").sum()
annual_costs = res.costs.resample("YE").sum()
pd.DataFrame({"turnover_per_year": annual_turnover, "cost_drag_per_year": annual_costs})

In [ ]:
equity_curve(bench)

In [ ]:
drawdown_chart(res.returns)

In [ ]:
heatmap_monthly(res.returns)

## Read

- Without leverage, risk parity will sit at lower vol and lower CAGR than 60/40 — the appeal is smaller drawdowns and steadier monthly returns.
- 2022 (stocks **and** bonds both fell, gold flat, DBC the only winner) is the regime stress-test. Watch how this period looks in the heatmap.
- Implementation: 5 ETFs in one taxable or IRA account, monthly trade.
- Variant: add `DBMF` as a 6th sleeve (managed-futures-style) — see notebook 06.